# nanochat d12 fp32 pretraining on Kaggle: 5 splits

This notebook runs the **d12 fp32 pretraining** stage of the `nanochat` pipeline on Kaggle 2xT4.

It is designed for five resumeable Kaggle runs. The tested fp32 settings are:

- `NANOCHAT_DTYPE=float32`
- `DEVICE_BATCH_SIZE=8`
- `TOTAL_BATCH_SIZE=524288`
- `TARGET_NUM_ITERATIONS=2520`
- no fp16 clamp path

## Before you run it

Make sure the Kaggle notebook is configured with:

- **GPU enabled** (`2x Tesla T4` expected)
- **Internet enabled**
- Your uploaded Kaggle dataset **`nanochat-codes`** attached
- For split runs 2-5, attach the previous split cache dataset listed below

The notebook auto-detects the attached code dataset from this mounted path pattern:
`/kaggle/input/datasets/<your-kaggle-username>/nanochat-codes`.

That mounted dataset is expected to contain the same project files that live under
`kaggle_dataset/nanochat/` in this repository.

## What this notebook does

1. Copy the attached `nanochat-codes` dataset into `/kaggle/working/nanochat`
2. Copy the previous split cache into `/kaggle/working/nanochat_cache` when needed
3. Configure caches under `/kaggle/working/nanochat_cache` and `/kaggle/working/huggingface_cache`
4. Install Python dependencies needed by the Kaggle runtime
5. Download the ClimbMix pretraining slice
6. Train/evaluate the tokenizer
7. Resume/train a `d12` fp32 base model to the configured split boundary
8. Optionally publish the pruned cache as a Kaggle Dataset for the next split


# Instructions for the D12 FP32 5-Split Process

This notebook uses `scripts.base_train_updated` for fp32 d12 training on Kaggle 2xT4. It uses five split runs of 504 optimizer steps each, for a total of 2520 steps.

Make sure the attached `nanochat-codes` Kaggle dataset has been refreshed after updating both `scripts/base_train_updated.py` and `nanochat/gpt.py`.

## Smoke run

In the first code cell, keep:

```python
PRETRAIN_SPLIT_INDEX = 0
```

This runs a short fp32 smoke and skips cache upload.

## Split runs

Use one notebook run per split. In the first code cell, set `PRETRAIN_SPLIT_INDEX` to the split you are running:

```python
PRETRAIN_SPLIT_INDEX = 1  # trains 0 -> 504
PRETRAIN_SPLIT_INDEX = 2  # resumes 504 -> 1008
PRETRAIN_SPLIT_INDEX = 3  # resumes 1008 -> 1512
PRETRAIN_SPLIT_INDEX = 4  # resumes 1512 -> 2016
PRETRAIN_SPLIT_INDEX = 5  # resumes 2016 -> 2520
```

Attach these resume datasets before each run:

- Split 1: attach no resume cache
- Split 2: attach `nanochat-d12-fp32-pretrain-cache-split-1`
- Split 3: attach `nanochat-d12-fp32-pretrain-cache-split-2`
- Split 4: attach `nanochat-d12-fp32-pretrain-cache-split-3`
- Split 5: attach `nanochat-d12-fp32-pretrain-cache-split-4`

The save/upload cell writes these dataset IDs:

- Split 1 output: `yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-1`
- Split 2 output: `yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-2`
- Split 3 output: `yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-3`
- Split 4 output: `yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-4`
- Split 5 final output: `yshuaiqin/nanochat-d12-fp32-pretrain-cache`

After split 2 succeeds, you can delete the split 1 dataset; after split 3 succeeds, delete split 2, and so on.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# 0 = smoke. 1..5 = actual fp32 training splits.
PRETRAIN_SPLIT_INDEX = 0
NUM_PRETRAIN_SPLITS = 5
assert PRETRAIN_SPLIT_INDEX in range(NUM_PRETRAIN_SPLITS + 1), (
    'PRETRAIN_SPLIT_INDEX must be 0 for smoke or 1..5 for split training'
)
PRETRAIN_RUN_MODE = 'smoke' if PRETRAIN_SPLIT_INDEX == 0 else f'split_{PRETRAIN_SPLIT_INDEX}'

RESUME_DATASET_NAME_BY_SPLIT = {
    2: 'nanochat-d12-fp32-pretrain-cache-split-1',
    3: 'nanochat-d12-fp32-pretrain-cache-split-2',
    4: 'nanochat-d12-fp32-pretrain-cache-split-3',
    5: 'nanochat-d12-fp32-pretrain-cache-split-4',
}
CACHE_DATASET_NAME = RESUME_DATASET_NAME_BY_SPLIT.get(PRETRAIN_SPLIT_INDEX)

DATASETS_ROOT = Path('/kaggle/input/datasets')
dataset_candidates = sorted(DATASETS_ROOT.glob('*/nanochat-codes'))
cache_candidates = sorted(DATASETS_ROOT.glob(f'*/{CACHE_DATASET_NAME}')) if CACHE_DATASET_NAME else []

assert dataset_candidates, (
    "Could not find an attached Kaggle dataset matching "
    "'/kaggle/input/datasets/<username>/nanochat-codes'"
)
assert len(dataset_candidates) == 1, (
    f"Expected exactly one attached 'nanochat-codes' dataset, found: {dataset_candidates}"
)
assert len(cache_candidates) <= 1, (
    f"Expected at most one attached '{CACHE_DATASET_NAME}' dataset, found: {cache_candidates}"
)
if PRETRAIN_SPLIT_INDEX > 1:
    assert cache_candidates, (
        f"Split {PRETRAIN_SPLIT_INDEX} must attach resume dataset '{CACHE_DATASET_NAME}'."
    )

CODE_INPUT = dataset_candidates[0]
CACHE_INPUT = cache_candidates[0] if cache_candidates else None

assert CODE_INPUT.exists(), f'Code dataset not found: {CODE_INPUT}'
print('Pretrain run mode:', PRETRAIN_RUN_MODE)
print('Code input:', CODE_INPUT)
print('Expected resume dataset:', CACHE_DATASET_NAME if CACHE_DATASET_NAME else 'none')
print('Resume cache input:', CACHE_INPUT if CACHE_INPUT is not None else 'not attached')
print('Top-level code files:', sorted(p.name for p in CODE_INPUT.iterdir()))


In [ ]:
WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)
updated_train_script = WORK_REPO / 'scripts' / 'base_train_updated.py'
assert updated_train_script.exists(), (
    'Missing scripts/base_train_updated.py in attached nanochat-codes dataset. '
    'Refresh the code dataset before running this updated notebook.'
)
if CACHE_INPUT is not None:
    shutil.copytree(CACHE_INPUT, WORK_CACHE, dirs_exist_ok=True)
    print('Copied resume cache into:', WORK_CACHE)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['PYTHONFAULTHANDLER'] = '1'

print('Working repo:', WORK_REPO)
print('Nanochat base dir:', WORK_CACHE)
print('HF cache:', HF_CACHE)
print('Repo contents:', sorted(p.name for p in WORK_REPO.iterdir()))


## D12 Run Settings

Defaults here are for a short fp32 feasibility smoke run on Kaggle 2xT4. The per-device and total batch sizes are intentionally smaller than the real pretraining run so this test can answer: does d12 fp32 fit, how slow is it, and does the forward/backward path stay finite?


In [ ]:
MODEL_DEPTH = 12
MODEL_TAG = 'd12_kaggle'
NUM_GPUS = 2
TRAIN_SHARDS = 64

MAX_SEQ_LEN = 1024
DEVICE_BATCH_SIZE = 8
TOTAL_BATCH_SIZE = 524288
TARGET_PARAM_DATA_RATIO = 12

# fp32 path: no fp16 clamp workaround.
LR_STABILITY_SCALE = 1.0
FP16_SAFE_MLP = False
FP16_SAFE_MLP_CLAMP = 128.0
FP16_SAFE_PROJ_CLAMP = 1024.0
FP16_SAFE_RESID_CLAMP = 128.0
BASE_EMBEDDING_LR = 0.3
BASE_UNEMBEDDING_LR = 0.008
BASE_MATRIX_LR = 0.02
BASE_SCALAR_LR = 0.5
EFFECTIVE_EMBEDDING_LR = BASE_EMBEDDING_LR * LR_STABILITY_SCALE
EFFECTIVE_UNEMBEDDING_LR = BASE_UNEMBEDDING_LR * LR_STABILITY_SCALE
EFFECTIVE_MATRIX_LR = BASE_MATRIX_LR * LR_STABILITY_SCALE
EFFECTIVE_SCALAR_LR = BASE_SCALAR_LR * LR_STABILITY_SCALE

try:
    PRETRAIN_SPLIT_INDEX
except NameError:
    PRETRAIN_SPLIT_INDEX = 0
NUM_PRETRAIN_SPLITS = 5
assert PRETRAIN_SPLIT_INDEX in range(NUM_PRETRAIN_SPLITS + 1)
PRETRAIN_RUN_MODE = 'smoke' if PRETRAIN_SPLIT_INDEX == 0 else f'split_{PRETRAIN_SPLIT_INDEX}'
SMOKE_NUM_ITERATIONS = 20
EXPECTED_D12_SCALING_PARAMS = 110_100_912
TARGET_NUM_ITERATIONS = (TARGET_PARAM_DATA_RATIO * EXPECTED_D12_SCALING_PARAMS) // TOTAL_BATCH_SIZE
assert TARGET_NUM_ITERATIONS % NUM_PRETRAIN_SPLITS == 0, 'Expected equal 5-way split'
SPLIT_SIZE = TARGET_NUM_ITERATIONS // NUM_PRETRAIN_SPLITS
if PRETRAIN_SPLIT_INDEX == 0:
    SPLIT_RESUME_STEP = None
    SPLIT_STOP_STEP = SMOKE_NUM_ITERATIONS
else:
    SPLIT_RESUME_STEP = (PRETRAIN_SPLIT_INDEX - 1) * SPLIT_SIZE
    SPLIT_STOP_STEP = PRETRAIN_SPLIT_INDEX * SPLIT_SIZE

OUTPUT_DATASET_ID_BY_SPLIT = {
    1: 'yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-1',
    2: 'yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-2',
    3: 'yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-3',
    4: 'yshuaiqin/nanochat-d12-fp32-pretrain-cache-split-4',
    5: 'yshuaiqin/nanochat-d12-fp32-pretrain-cache',
}
OUTPUT_DATASET_TITLE_BY_SPLIT = {
    1: 'nanochat d12 fp32 pretrain cache split 1',
    2: 'nanochat d12 fp32 pretrain cache split 2',
    3: 'nanochat d12 fp32 pretrain cache split 3',
    4: 'nanochat d12 fp32 pretrain cache split 4',
    5: 'nanochat d12 fp32 pretrain cache final',
}
OUTPUT_DATASET_ID = OUTPUT_DATASET_ID_BY_SPLIT.get(PRETRAIN_SPLIT_INDEX)

EVAL_DEVICE_BATCH_SIZE = 1
EVAL_SPLIT_TOKENS = 262144

world_tokens_per_microbatch = DEVICE_BATCH_SIZE * MAX_SEQ_LEN * NUM_GPUS
assert TOTAL_BATCH_SIZE % world_tokens_per_microbatch == 0, (
    'TOTAL_BATCH_SIZE must be divisible by '
    'DEVICE_BATCH_SIZE * MAX_SEQ_LEN * NUM_GPUS'
)

print('Model tag:', MODEL_TAG)
print('Depth:', MODEL_DEPTH)
print('Train shards:', TRAIN_SHARDS)
print('Pretrain run mode:', PRETRAIN_RUN_MODE)
print('Split index:', PRETRAIN_SPLIT_INDEX)
print('Split size:', SPLIT_SIZE)
print('Resume step:', SPLIT_RESUME_STEP)
print('Stop step:', SPLIT_STOP_STEP)
print('Output dataset:', OUTPUT_DATASET_ID if OUTPUT_DATASET_ID else 'none')
print('Tokens per microbatch:', world_tokens_per_microbatch)
print('Gradient accumulation steps:', TOTAL_BATCH_SIZE // world_tokens_per_microbatch)
print('Total batch tokens:', TOTAL_BATCH_SIZE)
print('LR stability scale:', LR_STABILITY_SCALE)
print('FP16 safe MLP:', FP16_SAFE_MLP)
print('Embedding LR:', EFFECTIVE_EMBEDDING_LR)
print('Unembedding LR:', EFFECTIVE_UNEMBEDDING_LR)
print('Matrix LR:', EFFECTIVE_MATRIX_LR)
print('Scalar LR:', EFFECTIVE_SCALAR_LR)
print('Target iterations:', TARGET_NUM_ITERATIONS)


## Optional Hugging Face Token

The default pretraining data is public, so this cell normally does not need a token.
Leave `USE_HF_TOKEN = False` for a normal run. Set it to `True` only if you need
authenticated Hugging Face access in your Kaggle session.


In [ ]:
USE_HF_TOKEN = False

if USE_HF_TOKEN and not os.environ.get('HF_TOKEN'):
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

if os.environ.get('HF_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN is set.')
else:
    print('Skipping HF token. Set USE_HF_TOKEN = True if you need one.')


## Install Dependencies

Install the Python packages that are needed in a fresh Kaggle runtime. Kaggle may
already provide some of them, but keeping this cell here makes the notebook easier
to rerun on a new image.


In [ ]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'pyarrow>=19.0.0',
    'python-dotenv>=1.2.1',
    'pyyaml>=6.0.0',
    'regex>=2025.9.1',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'scipy>=1.15.3',
    'tabulate>=0.9.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)


In [ ]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


## 1. Download a Small Pretraining Slice

This downloads `TRAIN_SHARDS` train shards plus the fixed validation shard into
`/kaggle/working/nanochat_cache/base_data_climbmix`. The d12 default is 64
train shards, up from the old d8 smoke-run value of 20.


In [ ]:
cmd = [
    sys.executable,
    '-m', 'nanochat.dataset',
    '-n', str(TRAIN_SHARDS),
    '-w', '4',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


## 2. Train the Tokenizer

Train the BPE tokenizer from the downloaded pretraining shards. The tokenizer is
saved under `/kaggle/working/nanochat_cache/tokenizer`.


In [ ]:
tokenizer_dir = WORK_CACHE / 'tokenizer'
tokenizer_ready = (tokenizer_dir / 'token_bytes.pt').exists()

if tokenizer_ready and CACHE_INPUT is not None:
    print('Tokenizer already present in attached cache; skipping tokenizer training.')
else:
    cmd = [
        sys.executable,
        '-m', 'scripts.tok_train',
    ]

    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, check=True)


## 3. Evaluate the Tokenizer

Run a small tokenizer compression check and compare the trained tokenizer with
GPT-2 and GPT-4 style tokenizers on a few text snippets.


In [ ]:
cmd = [
    sys.executable,
    '-m', 'scripts.tok_eval',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


## 4. Pretrain a d12 Base Model

This starts a `d12` base-model pretraining run and writes the checkpoint to
`/kaggle/working/nanochat_cache/base_checkpoints/d12_kaggle`.

The command keeps the Kaggle 2xT4 stability settings from the successful d8 run:

- `NANOCHAT_DTYPE=float32` to test whether d12 is numerically stable without fp16 limits
- `NANOCHAT_DISABLE_COMPILE=1` for Kaggle notebook stability
- optimizer communication settings for the Kaggle-safe distributed path
- NCCL settings that avoid unsupported Kaggle interconnect paths
- periodic train-time eval and sampling disabled so pretraining focuses on checkpoint creation
- training progress printed every 50 steps to keep Kaggle output readable


In [ ]:
env = os.environ.copy()
env['NANOCHAT_DTYPE'] = 'float32'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['NANOCHAT_TRAIN_PRINT_EVERY'] = '50'

cmd = [
    'torchrun',
    '--standalone',
    f'--nproc_per_node={NUM_GPUS}',
    '-m', 'scripts.base_train_updated',
    '--',
    f'--depth={MODEL_DEPTH}',
    f'--model-tag={MODEL_TAG}',
    '--run=dummy',
    f'--max-seq-len={MAX_SEQ_LEN}',
    '--window-pattern=L',
    f'--device-batch-size={DEVICE_BATCH_SIZE}',
    f'--total-batch-size={TOTAL_BATCH_SIZE}',
    f'--target-param-data-ratio={TARGET_PARAM_DATA_RATIO}',
    f'--lr-scale={LR_STABILITY_SCALE}',
    '--eval-every=-1',
    '--core-metric-every=-1',
    '--sample-every=-1',
    '--save-every=-1',
]

if FP16_SAFE_MLP:
    cmd.append('--fp16-safe-mlp')
    cmd.append(f'--fp16-safe-mlp-clamp={FP16_SAFE_MLP_CLAMP}')
    cmd.append(f'--fp16-safe-proj-clamp={FP16_SAFE_PROJ_CLAMP}')
    cmd.append(f'--fp16-safe-resid-clamp={FP16_SAFE_RESID_CLAMP}')

if PRETRAIN_RUN_MODE == 'smoke':
    cmd.append(f'--num-iterations={TARGET_NUM_ITERATIONS}')
    cmd.append(f'--stop-at-step={SMOKE_NUM_ITERATIONS}')
elif 1 <= PRETRAIN_SPLIT_INDEX <= NUM_PRETRAIN_SPLITS:
    cmd.append(f'--num-iterations={TARGET_NUM_ITERATIONS}')
    if SPLIT_RESUME_STEP and SPLIT_RESUME_STEP > 0:
        cmd.append(f'--resume-from-step={SPLIT_RESUME_STEP}')
    cmd.append(f'--stop-at-step={SPLIT_STOP_STEP}')
else:
    raise ValueError(f'Unknown PRETRAIN_RUN_MODE: {PRETRAIN_RUN_MODE}')

if SPLIT_RESUME_STEP and SPLIT_RESUME_STEP > 0:
    checkpoint_dir = WORK_CACHE / 'base_checkpoints' / MODEL_TAG
    required = [
        checkpoint_dir / f'model_{SPLIT_RESUME_STEP:06d}.pt',
        checkpoint_dir / f'meta_{SPLIT_RESUME_STEP:06d}.json',
        checkpoint_dir / f'optim_{SPLIT_RESUME_STEP:06d}_rank0.pt',
        checkpoint_dir / f'optim_{SPLIT_RESUME_STEP:06d}_rank1.pt',
    ]
    missing = [path for path in required if not path.exists()]
    assert not missing, 'Missing resume checkpoint files: ' + ', '.join(str(p) for p in missing)

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 5. Evaluate the d12 Base Model

This loads `base_checkpoints/d12_kaggle` and runs a reduced 2-GPU evaluation.
The settings are intentionally small enough for Kaggle:

- run `core`, `bpb`, and `sample`
- limit CORE examples with `--max-per-task=50`
- reduce BPB tokens with `--split-tokens=262144`

These numbers are useful for a smoke test and rough sanity check. They are not
intended as final benchmark results.


In [ ]:
RUN_BASE_EVAL = False

if not RUN_BASE_EVAL:
    print('Skipping base eval for fp32 smoke. Set RUN_BASE_EVAL = True to run it.')
else:
    env = os.environ.copy()
    env['NANOCHAT_DTYPE'] = 'float32'
    env['NANOCHAT_DISABLE_COMPILE'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['NCCL_P2P_DISABLE'] = '1'
    env['NCCL_IB_DISABLE'] = '1'

    cmd = [
        'torchrun',
        '--standalone',
        f'--nproc_per_node={NUM_GPUS}',
        '-m', 'scripts.base_eval',
        '--',
        '--eval', 'core,bpb,sample',
        '--model-tag', MODEL_TAG,
        '--max-per-task', '50',
        '--device-batch-size', str(EVAL_DEVICE_BATCH_SIZE),
        '--split-tokens', str(EVAL_SPLIT_TOKENS),
    ]

    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 6. Inspect Pretraining Artifacts

Print the cache size and a shallow file listing so you can confirm that data,
tokenizer files, reports, and base checkpoints were written where expected.


In [ ]:
print('Cache size:')
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)

print()
print('Top-level cache contents:')
subprocess.run(['find', str(WORK_CACHE), '-maxdepth', '2', '-type', 'f'], check=False)


## 7. Optional: Serve the d12 Base Model

Run this only after `base_checkpoints/d12_kaggle` exists. The cell is disabled by
default; set `RUN_SERVER = True` when you want to launch the web server.

This serves the pretrained base model, not an SFT chat model. Treat it as a
loading and generation smoke test rather than a polished assistant demo.


In [ ]:
# Optional: start a NanoChat web server for the pretrained base checkpoint.
# Set RUN_SERVER = True and run this cell when you want to serve the model.
RUN_SERVER = False

if not RUN_SERVER:
    print('Skipping server start. Set RUN_SERVER = True to launch it.')
else:
    subprocess.check_call([
        sys.executable,
        '-m', 'pip', 'install', '-q',
        'fastapi>=0.115.0',
        'pydantic>=2.0.0',
        'uvicorn[standard]>=0.30.0',
    ])

    try:
        server.terminate()
        server.wait(timeout=10)
        print('Stopped old server.')
    except NameError:
        print('No existing server variable found.')
    except Exception as e:
        print('Could not stop old server cleanly:', e)
        try:
            server.kill()
            print('Killed old server.')
        except Exception:
            pass

    env = os.environ.copy()
    env['NANOCHAT_DTYPE'] = 'float32'
    if FP16_SAFE_MLP:
        env['NANOCHAT_FP16_SAFE_MLP'] = '1'
        env['NANOCHAT_FP16_SAFE_MLP_CLAMP'] = str(FP16_SAFE_MLP_CLAMP)
        env['NANOCHAT_FP16_SAFE_PROJ_CLAMP'] = str(FP16_SAFE_PROJ_CLAMP)
        env['NANOCHAT_FP16_SAFE_RESID_CLAMP'] = str(FP16_SAFE_RESID_CLAMP)
    env['OMP_NUM_THREADS'] = '1'
    env['NCCL_P2P_DISABLE'] = '1'
    env['NCCL_IB_DISABLE'] = '1'

    cmd = [
        sys.executable,
        '-m', 'scripts.chat_web',
        '--source', 'base',
        '--model-tag', MODEL_TAG,
        '--num-gpus', '1',
        '--host', '0.0.0.0',
        '--port', '8000',
    ]

    print('Running:', ' '.join(cmd))
    server = subprocess.Popen(cmd, cwd=WORK_REPO, env=env)
    print('Server starting on http://127.0.0.1:8000')


## 8. Optional: Test the Server

Run this after the server has started. The cell is disabled by default; set
`TEST_SERVER = True` to send one streaming request to `http://127.0.0.1:8000`.

Because the checkpoint is only pretrained, output quality is not expected to be
chat-like. This cell only checks that the server can accept a request and stream tokens.


In [ ]:
# Optional: smoke-test the local web server.
# Set TEST_SERVER = True after the server has started.
TEST_SERVER = False

if not TEST_SERVER:
    print('Skipping server test. Set TEST_SERVER = True after starting the server.')
else:
    import json
    import time
    import requests

    BASE_URL = 'http://127.0.0.1:8000'

    for attempt in range(30):
        try:
            health = requests.get(f'{BASE_URL}/health', timeout=5)
            health.raise_for_status()
            print('Health:', health.json())
            break
        except Exception as e:
            if attempt == 29:
                raise RuntimeError('Server did not become healthy.') from e
            time.sleep(2)

    payload = {
        'messages': [
            {'role': 'user', 'content': 'Write one short sentence about machine learning.'}
        ],
        'temperature': 0.8,
        'top_k': 50,
        'max_tokens': 80,
    }

    print('Assistant:', end=' ', flush=True)
    with requests.post(f'{BASE_URL}/chat/completions', json=payload, stream=True, timeout=120) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue
            data = json.loads(line[len('data: '):])
            if data.get('done'):
                break
            print(data.get('token', ''), end='', flush=True)
    print()


## 9. Optional: Stop the Server

Use this when you are done testing the local server. Set `STOP_SERVER = True`
and run the cell to terminate the background process.


In [ ]:
# Optional: stop the server started above.
STOP_SERVER = False

if not STOP_SERVER:
    print('Skipping server stop. Set STOP_SERVER = True to stop it.')
else:
    try:
        server.terminate()
        server.wait(timeout=10)
        print('Server stopped.')
    except NameError:
        print('No server variable found.')
    except Exception as e:
        print('Could not stop server cleanly:', e)
        try:
            server.kill()
            print('Server killed.')
        except Exception:
            pass


## 10. Optional: Save the Pretraining Cache as a Kaggle Dataset

For split runs, this publishes a pruned `/kaggle/working/nanochat_cache` as the
resume dataset for the next run. Smoke mode skips upload.

Intermediate split datasets keep optimizer shards. The final split dataset keeps
only tokenizer files plus the final model/meta checkpoint.


In [ ]:
# Optional: save /kaggle/working/nanochat_cache as a Kaggle Dataset.
# Intermediate splits keep optimizer shards for resume.
# The final split keeps only tokenizer plus final model/meta checkpoint.
import json

CACHE_DIR = Path('/kaggle/working/nanochat_cache')
DATASET_ID = OUTPUT_DATASET_ID_BY_SPLIT.get(PRETRAIN_SPLIT_INDEX, '')
TITLE = OUTPUT_DATASET_TITLE_BY_SPLIT.get(PRETRAIN_SPLIT_INDEX, 'nanochat d12 fp32 pretrain cache')
VERSION_MESSAGE = (
    f'Save d12 fp32 split {PRETRAIN_SPLIT_INDEX}/{NUM_PRETRAIN_SPLITS} '
    f'at step {SPLIT_STOP_STEP} with {TRAIN_SHARDS} train shards'
)
PRUNE_FOR_RESUME_UPLOAD = True
KEEP_CHECKPOINT_STEP = SPLIT_STOP_STEP if PRETRAIN_SPLIT_INDEX > 0 else None
KEEP_OPTIMIZER_STATE = 0 < PRETRAIN_SPLIT_INDEX < NUM_PRETRAIN_SPLITS

if not DATASET_ID:
    print('Skipping upload. Split index 0 is smoke mode and is not meant for publishing a cache.')
else:
    assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
    assert CACHE_DIR.exists(), f'{CACHE_DIR} does not exist'
    assert any(CACHE_DIR.iterdir()), f'{CACHE_DIR} is empty'

    if PRUNE_FOR_RESUME_UPLOAD:
        if KEEP_CHECKPOINT_STEP is None:
            raise RuntimeError('Do not publish a resume cache from smoke mode. Run split 1 first.')

        data_dir = CACHE_DIR / 'base_data_climbmix'
        if data_dir.exists():
            shutil.rmtree(data_dir)
            print('Pruned downloaded pretraining data:', data_dir)

        checkpoint_dir = CACHE_DIR / 'base_checkpoints' / MODEL_TAG
        assert checkpoint_dir.exists(), f'Missing checkpoint dir: {checkpoint_dir}'

        def checkpoint_step(path):
            for part in path.stem.split('_'):
                if part.isdigit():
                    return int(part)
            return None

        for pattern in ['model_*.pt', 'meta_*.json']:
            for path in checkpoint_dir.glob(pattern):
                if checkpoint_step(path) != KEEP_CHECKPOINT_STEP:
                    path.unlink()
                    print('Pruned old checkpoint file:', path.name)

        for path in checkpoint_dir.glob('optim_*.pt'):
            if (not KEEP_OPTIMIZER_STATE) or checkpoint_step(path) != KEEP_CHECKPOINT_STEP:
                path.unlink()
                print('Pruned optimizer checkpoint file:', path.name)

        required = [
            checkpoint_dir / f'model_{KEEP_CHECKPOINT_STEP:06d}.pt',
            checkpoint_dir / f'meta_{KEEP_CHECKPOINT_STEP:06d}.json',
        ]
        if KEEP_OPTIMIZER_STATE:
            required.extend([
                checkpoint_dir / f'optim_{KEEP_CHECKPOINT_STEP:06d}_rank0.pt',
                checkpoint_dir / f'optim_{KEEP_CHECKPOINT_STEP:06d}_rank1.pt',
            ])
        missing = [path for path in required if not path.exists()]
        assert not missing, 'Missing cache files: ' + ', '.join(str(p) for p in missing)
        print('Kept checkpoint step:', KEEP_CHECKPOINT_STEP)
        print('Kept optimizer state:', KEEP_OPTIMIZER_STATE)

    metadata = {
        'title': TITLE,
        'id': DATASET_ID,
        'licenses': [
            {'name': 'CC0-1.0'}
        ]
    }

    metadata_path = CACHE_DIR / 'dataset-metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2))

    print('Dataset metadata:')
    print(metadata_path.read_text())

    print()
    print('Folder size:')
    subprocess.run(['du', '-sh', str(CACHE_DIR)], check=False)

    status = subprocess.run(
        ['kaggle', 'datasets', 'status', DATASET_ID],
        text=True,
        capture_output=True,
    )

    if status.returncode == 0:
        print()
        print(f'Dataset already exists: {DATASET_ID}')
        print('Creating a new dataset version...')
        cmd = [
            'kaggle', 'datasets', 'version',
            '-p', str(CACHE_DIR),
            '-m', VERSION_MESSAGE,
            '--dir-mode', 'tar',
        ]
    else:
        print()
        print(f'Dataset does not exist yet: {DATASET_ID}')
        print('Creating a new private dataset...')
        cmd = [
            'kaggle', 'datasets', 'create',
            '-p', str(CACHE_DIR),
            '--dir-mode', 'tar',
        ]

    print()
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True)

    if result.returncode != 0:
        raise RuntimeError('Kaggle Dataset upload failed.')

    print()
    print('Done.')
    print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')
